# Environment

In [ ]:
import pandas as pd
import numpy as np

# Data Loading

In [ ]:
master_table = pq.read_table(os.path.join(kb.source_tables_path, "tabla_madre/master_table.parquet")).to_pandas()
master_table.shape

# Quality Control

In [ ]:
selected_columns = ['DEMANDAPK'] + ['PATIENTID'] + ['IDTARJETASANITARIA', 'DNI', 'NOMBRE_APELLIDOS', 'FNACIMIENTO']
master_table[selected_columns].head(10)

In [ ]:
# To view all rows where the PATIENTID value is repeated
# We use keep=False to mark ALL instances (the first and all subsequent duplicates) as True
duplicate_patient_id_rows = master_table[
    master_table['PATIENTID'].duplicated(keep=False)
].sort_values(by='PATIENTID')

# Display the duplicate rows and the relevant linking columns
print("Rows with Duplicate PATIENTIDs:")

In [ ]:
# Select the columns you want to view for verification
cols_to_view = ['DEMANDAPK', 'PATIENTID', 'IDTARJETASANITARIA', 'DNI', 'NOMBRE_APELLIDOS', 'FNACIMIENTO']
duplicate_patient_id_rows[cols_to_view].tail(100)

## Analysis of missing 

In [ ]:
# ============================================================================
# 1. CREATE SUBSET OF UNLINKED RECORDS
# ============================================================================

def create_unlinked_subset(df, id_columns):
    """
    Creates a subset with records that have:
    - All 3 ID variables as null, OR
    - Only NOMBRE_APELLIDOS not null (P1 and P2 are None)
    
    Args:
        df: Original DataFrame
        id_columns: List with [IDTARJETASANITARIA, DNI, NOMBRE_APELLIDOS]
    
    Returns:
        DataFrame with unlinked records
    """
    
    p1_col = id_columns[0]  # IDTARJETASANITARIA
    p2_col = id_columns[1]  # DNI
    p3_col = id_columns[2]  # NOMBRE_APELLIDOS
    
    # Condition 1: All 3 IDs are null
    all_null = (df[p1_col].isna() & df[p2_col].isna() & df[p3_col].isna())
    
    # Condition 2: Only NOMBRE_APELLIDOS is not null (P1 and P2 are null)
    only_name = (df[p1_col].isna() & df[p2_col].isna() & df[p3_col].notna())
    
    # Combine both conditions
    mask_unlinked = all_null | only_name
    
    df_unlinked = df[mask_unlinked].copy()
    
    # Summary
    print(f"\n{'='*70}")
    print(f"SUBSET: UNLINKED RECORDS (P1 and P2)")
    print(f"{'='*70}")
    print(f"Total records in original dataset: {len(df):,}")
    print(f"Unlinked records: {len(df_unlinked):,} ({len(df_unlinked)/len(df)*100:.2f}%)")
    print(f"\nBreakdown:")
    print(f"  - With all 3 IDs null: {all_null.sum():,}")
    print(f"  - Only with NOMBRE_APELLIDOS: {only_name.sum():,}")
    print(f"{'='*70}\n")
    
    return df_unlinked


# ============================================================================
# 2. MISSING VALUES ANALYSIS
# ============================================================================

def analyze_missing_values(df):
    """
    Analyzes NaN values and blank strings in the DataFrame
    
    Args:
        df: DataFrame to analyze
    
    Returns:
        Tuple with (missing_table, blank_table, combined_table)
    """
    
    print(f"\n{'='*70}")
    print(f"MISSING VALUES ANALYSIS")
    print(f"{'='*70}\n")
    
    # --- NaN values analysis ---
    missing_percentages = df.isnull().mean() * 100
    missing_table = missing_percentages.reset_index()
    missing_table.columns = ['Variable', 'Porcentaje_NaN']
    missing_table = missing_table.sort_values(by='Porcentaje_NaN', ascending=False)
    
    print("📊 NaN VALUES BY VARIABLE:")
    print("-" * 70)
    print(missing_table.to_string(index=False))
    print()
    
    # --- Blank strings analysis ---
    blank_percentages = (df.apply(lambda col: col.map(
        lambda x: isinstance(x, str) and x.strip() == ''
    )).mean() * 100)
    
    blank_table = blank_percentages.reset_index()
    blank_table.columns = ['Variable', 'Porcentaje_en_blanco']
    blank_table = blank_table.sort_values(by='Porcentaje_en_blanco', ascending=False)
    
    print("📊 BLANK VALUES (empty strings) BY VARIABLE:")
    print("-" * 70)
    print(blank_table.to_string(index=False))
    print()
    
    # --- Combined summary ---
    combined = missing_table.merge(
        blank_table, 
        on='Variable', 
        how='outer'
    )
    combined['Total_Faltante'] = combined['Porcentaje_NaN'] + combined['Porcentaje_en_blanco']
    combined = combined.sort_values(by='Total_Faltante', ascending=False)
    
    print("📊 COMBINED SUMMARY (NaN + Blank):")
    print("-" * 70)
    print(combined.to_string(index=False))
    print(f"\n{'='*70}\n")
    
    return missing_table, blank_table, combined


# ============================================================================
# 3. COMPLETE USAGE EXAMPLE
# ============================================================================

# Create subset of unlinked records
df_unlinked = create_unlinked_subset(master_table, id_columns)

# Analyze missing values in the subset
missing_table, blank_table, combined_table = analyze_missing_values(df_unlinked)

# Optional: Save results
# df_unlinked.to_csv('unlinked_records.csv', index=False)
# combined_table.to_csv('missing_values_analysis.csv', index=False)

In [ ]:
df_unlinked.head(100)